# GRPO with verifiable rewards on MechaVerify

This notebook trains Qwen2.5-VL-3B-Instruct with GRPO, using the deterministic verifier as the reward (binary correctness plus a 0.1 format bonus). It then scores the adapter greedily on the in-distribution, OOD and adversarial splits of the released v0.1 set (56 items each, the same items as the SFT notebook). It runs on a single 16 GB GPU.

Settings follow the paper. Each prompt gets a group of G = 4 sampled responses, advantages come from within-group reward statistics, and a KL penalty (beta = 0.04) keeps the policy close to the reference. Training runs 150 steps at learning rate 1e-6 with completions of at most 256 tokens. Only the fp32 LoRA adapters train (the 4-bit base is frozen), so no mixed-precision loss scaler is used. Run the SFT notebook first with the same `SEED`, or set `SFT_ADAPTER = None` to train from the base model.

In [ ]:
SEED = 0        # 0, 1 or 2 (the paper reports all three)
SMOKE = False   # True runs a tiny end-to-end check instead of the full run
# adapter saved by the SFT notebook for this seed; None trains from the base model
SFT_ADAPTER = f"runs/{'smoke_' if SMOKE else ''}sft_s{SEED}/adapter"

In [ ]:
# Use this repository when the notebook runs from a clone, otherwise clone it.
import os, sys, subprocess
def _repo_root():
    for d in (os.getcwd(), os.path.abspath(".."), os.path.abspath("MechaVerify")):
        if os.path.isfile(os.path.join(d, "mechaverify", "__init__.py")):
            return d
    return None
REPO = _repo_root()
if REPO is None:
    REPO = os.path.abspath("MechaVerify")
    subprocess.run(["git", "clone", "--depth", "1",
                    "https://github.com/itswaseemsajjad/MechaVerify.git", REPO], check=True)
os.chdir(REPO)
if REPO not in sys.path:
    sys.path.insert(0, REPO)
%pip install -q -r requirements.txt -r requirements-train.txt

In [ ]:
import json, os
import torch
import transformers as _tf; _tf.set_seed(SEED)
from datasets import Dataset
from qwen_vl_utils import fetch_image
from mechaverify.dataset import generate_dataset, default_plans, SplitPlan, read_manifest
from mechaverify.tasks import list_tasks
from mechaverify.train.data import to_rl_records, make_reward_funcs, to_chat
from mechaverify.eval.runner import select_instances
from mechaverify.verify import verify

def log(*a): print("[mv]", *a, flush=True)

# The released v0.1 set (825 instances, seeds 1,000,000 to 1,000,780), regenerated from
# seeds. Training uses its train split. Every model is scored on the same fixed subset of
# its test splits: the first 8 items per task of id_test, ood_test and adversarial
# (n = 56 per split), the same items as `mechaverify evaluate --split <split> --per-task 8`.
if SMOKE:  # tiny end-to-end check
    plans = [SplitPlan("train", "mixed", [1, 2], 8),
             SplitPlan("id_test", "mixed", [1, 2], 3),
             SplitPlan("ood_test", "mixed", [5, 6], 3),
             SplitPlan("adversarial", "adv_idler", [2, 3], 3)]
    DATA, EVAL_PER_TASK = "data/mv_smoke", 2
else:
    plans = default_plans(60)
    DATA, EVAL_PER_TASK = "data/mv_release_v0_1", 8
card = generate_dataset(DATA, tasks=list_tasks(), plans=plans, seed0=1_000_000)
MANIFEST = DATA + "/manifest.jsonl"
if not SMOKE:  # the exact released set (manifest SHA-256 with LF line endings)
    import hashlib
    V01_SHA256 = "5db2d657facb6df25d3146f5e79298e58402821a46e291a4efe0c9fde762442b"
    _sha = hashlib.sha256(open(MANIFEST, "rb").read().replace(b"\r\n", b"\n")).hexdigest()
    assert card["n_instances"] == 825 and _sha == V01_SHA256, "the regenerated data differ from the released v0.1 set"
log("dataset:", card["n_instances"], card["counts_by_split"])
G, MAX_STEPS, MAXLEN = (2, 12, 256) if SMOKE else (4, 150, 256)
OUT = f"runs/{'smoke_' if SMOKE else ''}grpo{'' if SFT_ADAPTER else '_base'}_s{SEED}"
os.makedirs(OUT, exist_ok=True)

recs = to_rl_records(MANIFEST, split="train")

def _placeholders(messages):
    # TRL VLM-GRPO wants bare {"type":"image"} placeholders in the prompt; the
    # actual pixels come from the separate "image" column.
    out = []
    for m in messages:
        c = m["content"]
        if isinstance(c, list):
            c = [{"type": "image"} if (isinstance(p, dict) and p.get("type") == "image") else p
                 for p in c]
        out.append({"role": m["role"], "content": c})
    return out

# fetch_image applies the same resizing as process_vision_info in SFT and evaluation
rl_ds = Dataset.from_list([{"prompt": _placeholders(r["messages"]),
                            "image": fetch_image({"image": r["image"]}),
                            "answer": r["answer"]} for r in recs])
log("rl prompts:", len(rl_ds))

In [ ]:
from transformers import (Qwen2_5_VLForConditionalGeneration, AutoProcessor,
                          BitsAndBytesConfig)
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training, PeftModel

MODEL_ID = "Qwen/Qwen2.5-VL-3B-Instruct"
# QLoRA: 4-bit NF4 with double quantization, fp16 compute (bf16 yields NaN generations
# on pre-Ampere GPUs). The same loading is used for training and for evaluation.
CDT = torch.float16
bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",
                         bnb_4bit_use_double_quant=True, bnb_4bit_compute_dtype=CDT)
MAX_PIXELS = 512*28*28  # image resolution cap
# LoRA on the language-model projections only; the vision encoder stays frozen.
LORA_TARGETS = r"^(?!.*visual).*\.(q_proj|k_proj|v_proj|o_proj|gate_proj|up_proj|down_proj)$"
LORA = LoraConfig(r=16, lora_alpha=32, lora_dropout=0.05, bias="none",
                  task_type="CAUSAL_LM", target_modules=LORA_TARGETS)
LORA_PARAMS = 29_933_568  # rank-16 LoRA on the 36 language-model layers, nothing else
# evaluation: plain greedy decoding (no repetition penalty) with a 1536-token cap
MAX_NEW_TOKENS = 256 if SMOKE else 1536
GEN = dict(max_new_tokens=MAX_NEW_TOKENS, do_sample=False, repetition_penalty=1.0)
log("GPU memory (GB):", round(torch.cuda.get_device_properties(0).total_memory / 2**30, 1)
    if torch.cuda.is_available() else "no GPU")
model = Qwen2_5_VLForConditionalGeneration.from_pretrained(
    MODEL_ID, quantization_config=bnb, torch_dtype=CDT, device_map="auto")
processor = AutoProcessor.from_pretrained(MODEL_ID, min_pixels=256*28*28, max_pixels=MAX_PIXELS)

model = prepare_model_for_kbit_training(model)
model.config.use_cache = False
# Continue from the SFT adapter of the same seed (SFT -> RLVR), or train a fresh LoRA
# from the base model when SFT_ADAPTER is None.
if SFT_ADAPTER:
    if not os.path.exists(os.path.join(SFT_ADAPTER, "adapter_model.safetensors")):
        raise FileNotFoundError(f"no SFT adapter at {SFT_ADAPTER}; run the SFT notebook first "
                                "or set SFT_ADAPTER = None")
    model = PeftModel.from_pretrained(model, SFT_ADAPTER, is_trainable=True)
    INIT = f"SFT ({SFT_ADAPTER})"
else:
    _tf.set_seed(SEED)  # LoRA initialization
    model = get_peft_model(model, LORA)
    INIT = "base"
log("GRPO init from:", INIT)
model.print_trainable_parameters()
n_lora = sum(p.numel() for p in model.parameters() if p.requires_grad)
assert n_lora == LORA_PARAMS, f"{n_lora} trainable parameters, expected LoRA on the language model only"

In [ ]:
from trl import GRPOConfig, GRPOTrainer
cfg = GRPOConfig(
    output_dir=OUT, per_device_train_batch_size=1,
    gradient_accumulation_steps=4, num_generations=G, max_completion_length=MAXLEN,
    temperature=1.0, beta=0.04, learning_rate=1e-6, max_steps=MAX_STEPS,
    loss_type="grpo", logging_steps=2, save_strategy="no",
    # No fp16/bf16 AMP: only the fp32 LoRA adapters train (the 4-bit base is frozen),
    # so no mixed-precision loss scaler is needed.
    bf16=False, fp16=False,
    gradient_checkpointing=True, report_to="none", log_completions=False,
    seed=SEED, data_seed=SEED)
# GRPOTrainer casts the trainable adapter weights of a 4-bit model to bf16. Keep an
# exact fp32 copy and restore it, so the LoRA adapters train in fp32.
fp32 = {n: p.detach().clone() for n, p in model.named_parameters() if p.requires_grad}
trainer = GRPOTrainer(model=model, reward_funcs=make_reward_funcs(0.1), args=cfg,
                      train_dataset=rl_ds, processing_class=processor)
for n, p in model.named_parameters():
    if n in fp32:
        p.data = fp32[n]
assert all(p.dtype == torch.float32 for p in model.parameters() if p.requires_grad)
trainer.train()
model.save_pretrained(f"{OUT}/adapter", selected_adapters=["default"])
log("saved GRPO adapter ->", f"{OUT}/adapter")

In [ ]:
# Evaluate per split (greedy) with the deterministic verifier.
from qwen_vl_utils import process_vision_info
# Leave train mode and gradient checkpointing and re-enable the cache before
# generating, or the just-trained model yields NaN/garbage tokens.
model.eval()
try:
    model.gradient_checkpointing_disable()
except Exception as e:
    log("grad-ckpt disable:", e)
model.config.use_cache = True
@torch.no_grad()
def gen(inst):
    rec = to_chat(inst, DATA, include_target=False)
    text = processor.apply_chat_template(rec["messages"], tokenize=False, add_generation_prompt=True)
    imgs, _ = process_vision_info(rec["messages"])
    enc = processor(text=[text], images=imgs, return_tensors="pt").to(model.device)
    out = model.generate(**enc, **GEN)
    return processor.batch_decode(out[:, enc["input_ids"].shape[1]:], skip_special_tokens=True)[0]

results = {}
allins = read_manifest(MANIFEST)
for split in ("id_test", "ood_test", "adversarial"):
    insts = select_instances(allins, split=split, per_task=EVAL_PER_TASK)
    correct = sum(verify(i, gen(i)).correct for i in insts)
    results[split] = {"acc": correct / len(insts) if insts else 0.0, "n": len(insts)}
    log(f"eval {split}: {results[split]}")

json.dump({"smoke": SMOKE, "seed": SEED, "model": MODEL_ID, "init": INIT,
           "G": G, "max_steps": MAX_STEPS, "results": results},
          open(f"{OUT}/grpo_eval.json", "w"), indent=2)
log("DONE ->", f"{OUT}/grpo_eval.json")

## Mean and spread over seeds

After both notebooks have run for seeds 0, 1 and 2 in the same clone, this cell prints the per-split mean and sample standard deviation, the in-distribution minus OOD gap of the means, the SFT perception gap, and the zero-shot base-model accuracy.

In [ ]:
import glob, json, statistics

SPLITS = ("id_test", "ood_test", "adversarial")

def summarize(pattern):
    runs = [json.load(open(p))["results"] for p in sorted(glob.glob(pattern))]
    if len(runs) < 2:
        return f"{len(runs)} run(s) found"
    mean = {s: statistics.mean(r[s]["acc"] for r in runs) for s in SPLITS}
    std = {s: statistics.stdev(r[s]["acc"] for r in runs) for s in SPLITS}
    out = {s: f"{mean[s]:.3f} ± {std[s]:.3f}" for s in SPLITS}
    out["gap (in-dist - OOD)"] = round(mean["id_test"] - mean["ood_test"], 3)
    out["n_seeds"] = len(runs)
    return out

def summarize_gap(pattern):
    runs = [json.load(open(p)).get("sft") for p in sorted(glob.glob(pattern))]
    runs = [r for r in runs if r]
    if len(runs) < 2:
        return f"{len(runs)} run(s) found"
    return {s: f"{statistics.mean(r[s + '_gap'] for r in runs):+.3f} ± "
               f"{statistics.stdev(r[s + '_gap'] for r in runs):.3f}" for s in SPLITS}

print("SFT                       ", summarize("runs/sft_s*/sft_eval.json"))
print("SFT -> RLVR               ", summarize("runs/grpo_s*/grpo_eval.json"))
print("SFT perception gap (S - D)", summarize_gap("runs/sft_s*/perception_eval.json"))
base = [b for b in (json.load(open(p)).get("base") for p in sorted(glob.glob("runs/sft_s*/perception_eval.json"))) if b]
if base:
    print("Base model, condition D   ", {s: base[0][f"{s}_D"] for s in SPLITS})